# asset-forge batch GPU notebook

Runs heavy 3D-generation jobs on Kaggle's free GPU quota.
Designed to be uploaded as a new version by asset-forge's
`KaggleBatchBackend` via the Kaggle API.

**Inputs**: a Kaggle Dataset containing `job_spec.json` and any
reference images.

**Outputs**: GLB files in `/kaggle/working/out/` for asset-forge
to download via `kaggle kernels output`.

Supported models (selectable via `job_spec.json`):
- `hunyuan3d-full`  Tencent Hunyuan3D 2.1 full (needs ~10 GB VRAM, P100 OK)
- `trellis-local`   Microsoft TRELLIS (needs ~10 GB VRAM)
- `controlnet-style`  Style-consistency regen pass
- `anigen`          Auto-rigging from single image (~12 GB, P100 borderline)

All four are too heavy for the operator's 6 GB local GPU but fit
P100 16 GB / T4x2.

In [ ]:
import json
import os
import sys
import subprocess
from pathlib import Path

# Locate the input dataset asset-forge uploaded
input_root = Path('/kaggle/input')
datasets = list(input_root.iterdir())
assert datasets, 'No input dataset attached to this notebook'
spec_path = next((d / 'job_spec.json' for d in datasets if (d / 'job_spec.json').exists()), None)
assert spec_path, 'job_spec.json not found in input dataset'

spec = json.loads(spec_path.read_text())
print(f'Loaded job spec: model={spec["model"]}, pieces={len(spec["pieces"])}')

out_root = Path('/kaggle/working/out')
out_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU on this Kaggle runtime'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')

In [ ]:
# Dispatch to model-specific cell. Only one of these runs per notebook execution.
MODEL_DISPATCH = {
    'hunyuan3d-full': 'hunyuan3d_full.py',
    'trellis-local': 'trellis_local.py',
    'controlnet-style': 'controlnet_style.py',
    'anigen': 'anigen.py',
}

model = spec['model']
if model not in MODEL_DISPATCH:
    raise ValueError(f'Unknown model: {model}. Supported: {list(MODEL_DISPATCH)}')

# Scripts ship as additional dataset assets next to job_spec.json
script_path = spec_path.parent / MODEL_DISPATCH[model]
print(f'Dispatching to: {script_path}')

# Execute the model script with spec path + output root
result = subprocess.run(
    [sys.executable, str(script_path), str(spec_path), str(out_root)],
    capture_output=True, text=True, timeout=spec.get('timeout_minutes', 60) * 60,
)
print(result.stdout[-4000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-4000:])
    raise RuntimeError(f'Model script failed with exit code {result.returncode}')

In [ ]:
# Final summary: list outputs for asset-forge to download
outputs = sorted(out_root.rglob('*.glb'))
summary = {
    'job_id': spec.get('job_id'),
    'model': spec['model'],
    'pieces_requested': len(spec['pieces']),
    'pieces_produced': len(outputs),
    'outputs': [str(p.relative_to(out_root)) for p in outputs],
    'gpu': gpu_name,
    'vram_gb': vram_gb,
}
(out_root / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))